# BODYFRAME Stage 1 — File Intake & Metadata QA

This notebook is a lightweight interface to `intake_validation.py`. It performs a read-only validation unless an explicit noncanonical output is requested through the script CLI.

## 1. Environment and imports

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

## 2. Project paths

In [ ]:
working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir if (working_dir / 'data').exists() else working_dir.parent
STAGE1_DIR = PROJECT_ROOT / '01_file_intake'
REGISTRY_PATH = PROJECT_ROOT / 'data' / 'raw_tables' / 'bodyframe_master_registry.xlsx'
CANONICAL_MANIFEST_PATH = STAGE1_DIR / 'bodyframe_file_manifest.xlsx'
sys.path.insert(0, str(STAGE1_DIR))

from intake_validation import (
    apply_duplicate_detection,
    compare_with_canonical,
    finalize_validation,
    inspect_assets,
    load_canonical_manifest,
    load_registry_assets,
    reproducibility_checks,
    summarize_results,
)

PROJECT_ROOT

## 3. Registry load

In [ ]:
registry_assets = load_registry_assets(REGISTRY_PATH)
print(f'Registered assets: {len(registry_assets)}')

## 4. File inspection

In [ ]:
results = inspect_assets(registry_assets, PROJECT_ROOT)
print(f'Files inspected: {len(results)}')

## 5. Validation

In [ ]:
apply_duplicate_detection(results)
finalize_validation(results)
reproduction = reproducibility_checks(results)
pprint(reproduction)

## 6. Duplicate analysis

In [ ]:
duplicate_rows = [
    {
        'asset_id': row['asset_id'],
        'exact_duplicate': row['exact_duplicate'],
        'duplicate_group_id': row['duplicate_group_id'],
    }
    for row in results
    if row['duplicate_group_id']
]
pprint(duplicate_rows)

## 7. Summary

In [ ]:
summary = summarize_results(results)
canonical_rows = load_canonical_manifest(CANONICAL_MANIFEST_PATH)
comparison = compare_with_canonical(results, canonical_rows)
pprint(summary)
pprint(comparison)